# Fase 8 — Detektor Pembanding (DSF-YOLO) — **SATU MODEL PER RUN**

Melatih detektor pembanding dgn **resep IDENTIK** & **framework sama** (Ultralytics 8.2.29
yang dibundel repo Mamba-YOLO) seperti Mamba-YOLO-T dan DeformSS2D.

| Model | Params (nc=8) | Peran |
|---|---|---|
| YOLOv3-tiny | 12.136.240 | arsitektur lama |
| YOLOv5s | 9.125.288 | klasik, banyak disitasi |
| YOLOv6n | 4.238.936 | tepat di bawah skala kita |
| YOLOv8n | 3.012.408 | ringan modern |
| YOLOv8s | 11.138.696 | modern, di atas skala kita |
| *(internal)* | *B2 5.986.912 · P1 5.930.869* | |

**Semua dilatih DARI NOL** (dari `.yaml`, bukan bobot pra-latih) — sama seperti model kita.

⚠️ `lr0=0.002` mengikuti protokol kita, lebih rendah dari default Ultralytics (0.01).
Pembanding mungkin belum optimal di setelan ini → **wajib dinyatakan di naskah**.

**Cara pakai:** ubah `MODEL_KEY` di Sel 3, lalu *Run all*. Ulangi untuk model berikutnya.
Auto-resume kalau sesi terputus.

## 1. Mount Drive + Kredensial Kaggle

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, shutil, subprocess
DRIVE_ROOT   = "/content/drive/MyDrive/demamba-yolowd"
DRIVE_OUTPUT = f"{DRIVE_ROOT}/output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True); os.makedirs(f"{DRIVE_ROOT}/analysis", exist_ok=True)

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy(f"{DRIVE_ROOT}/kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
print("Drive + Kaggle siap")

Mounted at /content/drive
Drive + Kaggle siap


## 2. Tarik Dataset + Deteksi GPU

In [2]:
import torch
DATA_DIR = "/content/dsf"
if not glob.glob(f"{DATA_DIR}/**/*.txt", recursive=True):
    os.makedirs(DATA_DIR, exist_ok=True)
    subprocess.run(["kaggle","datasets","download","-d","benyxu/dsf-yolo-steel-tube-dataset",
                    "-p",DATA_DIR,"--unzip"], check=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA")
nw = min(os.cpu_count() // max(torch.cuda.device_count(), 1), 8)
print(f"cpu_count={os.cpu_count()} -> dataloader workers={nw}")
print("CATAT angka workers ini: ia menentukan aliran augmentasi & harus sama dgn run Mamba-YOLO kita.")
assert torch.cuda.is_available(), "GPU tidak aktif -- Runtime > Change runtime type"

GPU: NVIDIA A100-SXM4-40GB
cpu_count=12 -> dataloader workers=8
CATAT angka workers ini: ia menentukan aliran augmentasi & harus sama dgn run Mamba-YOLO kita.


## 3. 🔧 KONFIGURASI — **Ubah `MODEL_KEY` Setiap Ganti Model**

In [3]:
# ==========================================================
#  UBAH DUA BARIS INI SEBELUM "Run all"
MODEL_KEY = "YOLOv8s"      # YOLOv3-tiny | YOLOv5s | YOLOv6n | YOLOv8n | YOLOv8s
OPTIMIZER = "auto"         # "auto"  -> resep wajar framework (setara B2-auto)
                           # "SGD"   -> resep terkunci kita (lr0=0.002)
# ==========================================================
SEED, TARGET_EPOCHS, LR0, USE_COS_LR = 1, 150, 0.002, False

MODELS = {                       # label -> (yaml, params yang diharapkan pd nc=8)
    "YOLOv3-tiny": ("yolov3-tiny.yaml", 12_136_240),
    "YOLOv5s":     ("yolov5s.yaml",      9_125_288),
    "YOLOv6n":     ("yolov6n.yaml",      4_238_936),
    "YOLOv8n":     ("yolov8n.yaml",      3_012_408),
    "YOLOv8s":     ("yolov8s.yaml",     11_138_696),
}
assert MODEL_KEY in MODELS, f"MODEL_KEY harus salah satu dari {list(MODELS)}"
assert OPTIMIZER in ("auto", "SGD"), 'OPTIMIZER harus "auto" atau "SGD"'
MODEL_YAML, EXPECTED_PARAMS = MODELS[MODEL_KEY]

AUTO = OPTIMIZER == "auto"
MODEL_LABEL = f"{MODEL_KEY}-auto" if AUTO else MODEL_KEY          # label BEDA -> baris CSV tidak bertabrakan
RUN_NAME    = (f"{MODEL_KEY.lower()}_seed{SEED}_{TARGET_EPOCHS}ep_auto" if AUTO
               else f"{MODEL_KEY.lower()}_seed{SEED}_{TARGET_EPOCHS}ep_lr{LR0}")
OUT_NAME    = "per_class_results_recipe.csv" if AUTO else "per_class_results_baselines.csv"

print(f"MODEL      : {MODEL_KEY}  ({MODEL_YAML})   label CSV: {MODEL_LABEL}")
print(f"RUN_NAME   : {RUN_NAME}")
print(f"CSV        : {OUT_NAME}")
if AUTO:
    print("resep      : optimizer=auto -> framework memilih sendiri; lr0 & momentum DIABAIKAN")
    print("             utk dataset ini: iterations=6450 <= 10000 -> AdamW lr=0.000833, momentum=0.9")
    print("             pembanding: B2-auto = 0.6796 | B2 resep terkunci = 0.5662 +/- 0.0147")
else:
    print(f"resep      : SGD lr0={LR0} cos_lr={USE_COS_LR}")
print(f"tetap sama : {TARGET_EPOCHS} epoch, batch 8, 640 px, patience=0, cos_lr={USE_COS_LR}, TF32 mati, seed {SEED}")

MODEL      : YOLOv8s  (yolov8s.yaml)   label CSV: YOLOv8s-auto
RUN_NAME   : yolov8s_seed1_150ep_auto
CSV        : per_class_results_recipe.csv
resep      : optimizer=auto -> framework memilih sendiri; lr0 & momentum DIABAIKAN
             utk dataset ini: iterations=6450 <= 10000 -> AdamW lr=0.000833, momentum=0.9
             pembanding: B2-auto = 0.6796 | B2 resep terkunci = 0.5662 +/- 0.0147
tetap sama : 150 epoch, batch 8, 640 px, patience=0, cos_lr=False, TF32 mati, seed 1


## 4. Pasang Ultralytics 8.2.29 (paket RESMI) + Patch `torch.load`

**Tanpa repo Mamba-YOLO dan tanpa kernel `selective_scan`.** Diverifikasi offline bahwa paket
resmi 8.2.29 memberi model **identik parameter demi parameter** dgn fork yang dipakai run kita,
dan baris penentu urutan data (`generator.manual_seed(6148914691236517205 + RANK)`) **sama persis**.
Satu-satunya beda fork: tambahan modul Mamba, satu list-comprehension di fungsi model World, dan
satu impor tak terpakai — tidak ada yang menyentuh training deteksi.

Patch `torch.load` tetap perlu: torch baru memakai `weights_only=True` sebagai default, yang
membuat Ultralytics 8.2.29 gagal menyimpan/memuat checkpoint.

In [4]:
subprocess.run([sys.executable,"-m","pip","install","-q","ultralytics==8.2.29"], check=True)
for k in [x for x in list(sys.modules) if x.startswith("ultralytics")]: del sys.modules[k]
import ultralytics, py_compile
PKG = os.path.dirname(ultralytics.__file__)
print("ultralytics", ultralytics.__version__, "->", PKG)
assert ultralytics.__version__ == "8.2.29", "versi harus 8.2.29 agar setara run Mamba-YOLO kita"

def patch(relpath, subs):
    p = os.path.join(PKG, relpath); src = open(p).read()
    for old, new, tag in subs:
        if new in src: print(f"  ~ {tag}: skip"); continue
        assert src.count(old) == 1, f"GAGAL {tag}: {src.count(old)}x (harus 1x)"
        src = src.replace(old, new); print(f"  OK {tag}")
    open(p, "w").write(src); py_compile.compile(p, doraise=True)

print("### Patch torch.load")
patch("nn/tasks.py", [
    ('            ckpt = torch.load(file, map_location="cpu")\n\n    except ModuleNotFoundError as e:',
     '            ckpt = torch.load(file, map_location="cpu", weights_only=False)\n\n    except ModuleNotFoundError as e:', "4.1"),
    ('        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu")',
     '        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu", weights_only=False)', "4.2"),
])
patch("utils/torch_utils.py", [
    ('    x = torch.load(f, map_location=torch.device("cpu"))\n    if "model" not in x:',
     '    x = torch.load(f, map_location=torch.device("cpu"), weights_only=False)\n    if "model" not in x:', "5.1"),
])

subprocess.run([sys.executable,"-m","pip","install","-q","seaborn","thop"], check=True)
subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q",
                "wandb","ray","tensorboard","tensorboardX","tensorboard-data-server",
                "clearml","comet_ml","dvclive","mlflow"], check=False)
for k in [x for x in list(sys.modules) if x.startswith("ultralytics")]: del sys.modules[k]
print("Lingkungan siap -- tanpa Mamba, tanpa kernel CUDA tambahan")

ultralytics 8.2.29 -> /usr/local/lib/python3.13/dist-packages/ultralytics
### Patch torch.load
  OK 4.1
  OK 4.2
  OK 5.1
Lingkungan siap -- tanpa Mamba, tanpa kernel CUDA tambahan


## 5. data.yaml — dibuat sendiri (dataset tidak menyertakannya)

Isi & urutan kelas **sama persis** dgn yang dipakai ke-40 run DSF sebelumnya.

In [5]:
import yaml
from pathlib import Path

# Dataset DSF-YOLO TIDAK menyertakan data.yaml -- notebook Mamba-YOLO kita membuatnya sendiri.
# Isi di bawah SAMA PERSIS dgn yang dipakai ke-40 run DSF (fase-2 sel 8): struktur train2021/
# val2021 dan urutan 8 kelas. Urutan nama = indeks kelas; JANGAN diubah, per-kelas jadi salah.
cands = [p.parent.parent for p in Path(DATA_DIR).rglob("images/train2021") if p.is_dir()]
assert cands, f"struktur images/train2021 tidak ditemukan di {DATA_DIR} -- cek isi dataset"
DSF_BASE = str(cands[0].resolve())

NAMES = ["air-hole", "bite-edge", "broken-arc", "crack",
         "dense-gas-hole", "non-fusion", "slag-inclusion", "unfused-hole"]
DATA_YAML = "/content/dsf-yolo.yaml"
yaml.safe_dump({"path": DSF_BASE, "train": "images/train2021", "val": "images/val2021",
                "nc": len(NAMES), "names": NAMES},
               open(DATA_YAML, "w"), sort_keys=False, default_flow_style=False)

for split in ["train2021", "val2021"]:
    d = Path(DSF_BASE) / "images" / split
    n_img = len([f for f in d.iterdir() if f.suffix.lower() in {".jpg",".jpeg",".png",".bmp"}])
    lbl = Path(DSF_BASE) / "labels" / split
    n_lbl = len(list(lbl.glob("*.txt"))) if lbl.is_dir() else 0
    print(f"  {split}: {n_img} citra, {n_lbl} label")
assert (Path(DSF_BASE)/"labels"/"train2021").is_dir(), "folder labels/ tidak ditemukan"
print(f"\nDSF_BASE : {DSF_BASE}")
print(f"kelas ({len(NAMES)}): {NAMES}")
print(f"data.yaml: {DATA_YAML}")

  train2021: 2726 citra, 2726 label
  val2021: 682 citra, 682 label

DSF_BASE : /content/dsf/steel-tube-dataset-all/yolo
kelas (8): ['air-hole', 'bite-edge', 'broken-arc', 'crack', 'dense-gas-hole', 'non-fusion', 'slag-inclusion', 'unfused-hole']
data.yaml: /content/dsf-yolo.yaml


## 6. Pre-flight — verifikasi arsitektur sebelum menghabiskan GPU

In [6]:
from copy import deepcopy
from ultralytics import YOLO
from ultralytics.nn.tasks import parse_model, yaml_model_load

d = yaml_model_load(MODEL_YAML); d["nc"] = len(NAMES)
m, _ = parse_model(deepcopy(d), ch=3, verbose=False)
n_params = sum(p.numel() for p in m.parameters())
print(f"{MODEL_KEY}: {n_params:,} parameter (harapan {EXPECTED_PARAMS:,})")
assert n_params == EXPECTED_PARAMS, "jumlah parameter TIDAK cocok -- versi framework berubah? hentikan."
del m
print("Pre-flight OK")

YOLOv8s: 11,138,696 parameter (harapan 11,138,696)
Pre-flight OK


## 7. Cek Status Checkpoint — Baru / Belum Selesai / Sudah Selesai

In [7]:
last_ckpt = Path(DRIVE_OUTPUT) / RUN_NAME / "weights" / "last.pt"
if not last_ckpt.exists():
    STATUS = "BARU"; print(f"STATUS: {STATUS} -- belum ada checkpoint, training mulai dari nol.")
else:
    raw = torch.load(str(last_ckpt), map_location="cpu", weights_only=False)
    ep, has_opt = raw.get("epoch", -1), raw.get("optimizer") is not None
    if has_opt and 0 <= ep < TARGET_EPOCHS - 1:
        STATUS = "BELUM_SELESAI"
        print(f"STATUS: {STATUS} -- epoch tersimpan={ep} -> akan RESUME dari epoch {ep+1}.")
    else:
        STATUS = "SELESAI"
        print(f"STATUS: {STATUS} (optimizer={has_opt}, epoch={ep}) -- training TIDAK diulang.")
print(f"\nlast.pt: {last_ckpt}  (ada: {last_ckpt.exists()})")

STATUS: BARU -- belum ada checkpoint, training mulai dari nol.

last.pt: /content/drive/MyDrive/demamba-yolowd/output/yolov8s_seed1_150ep_auto/weights/last.pt  (ada: False)


## 8. Training — mulai baru atau resume, sesuai Sel 7

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = False      # sama spt seluruh run kita
torch.backends.cudnn.allow_tf32 = False

if STATUS == "BARU":
    kw = dict(
        data=DATA_YAML, epochs=TARGET_EPOCHS,
        batch=8, imgsz=640, device=0, amp=True, seed=SEED,
        patience=0,          # WAJIB: matikan EarlyStopping
        cache=True,          # percepat; tidak mengubah data/hasil
        optimizer=OPTIMIZER,
        cos_lr=USE_COS_LR,
        deterministic=True, workers=8,
        project=DRIVE_OUTPUT, name=RUN_NAME, plots=False,
    )
    if not AUTO:
        kw["lr0"] = LR0      # hanya bermakna utk SGD; "auto" mengabaikannya
    model = YOLO(MODEL_YAML)          # dari .yaml = DARI NOL (bukan bobot pra-latih)
    model.train(**kw)
elif STATUS == "BELUM_SELESAI":
    model = YOLO(str(last_ckpt))
    model.train(resume=True)          # train_args diambil dari checkpoint
else:
    model = YOLO(str(last_ckpt))
    print("Training sudah selesai sebelumnya -- sel ini di-skip.")

SAVE_DIR = model.trainer.save_dir if getattr(model, "trainer", None) else Path(DRIVE_OUTPUT)/RUN_NAME
print(f"\nSAVE_DIR: {SAVE_DIR}")
if AUTO:
    print("PENTING: cari baris 'optimizer:' di log & CATAT optimizer + lr yang dipilih framework.")
fin = torch.load(str(Path(SAVE_DIR)/"weights"/"last.pt"), map_location="cpu", weights_only=False)
print("final_eval selesai" if fin.get("optimizer") is None else f"BELUM tuntas (epoch={fin.get('epoch')}) -- jalankan ulang notebook utk resume")

New https://pypi.org/project/ultralytics/8.4.165 available 😃 Update with 'pip install -U ultralytics'
New https://pypi.org/project/ultralytics/8.4.165 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.yaml, data=/content/dsf-yolo.yaml, epochs=150, time=None, patience=0, batch=8, imgsz=640, save=True, save_period=-1, cache=True, device=0, workers=8, project=/content/drive/MyDrive/demamba-yolowd/output, name=yolov8s_seed1_150ep_auto, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=1, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, 

100%|██████████| 755k/755k [00:00<00:00, 15.4MB/s]

Overriding model.yaml nc=80 with nc=8
Overriding model.yaml nc=80 with nc=8

                   from  n    params  module                                       arguments                     

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3       

YOLOv8s summary: 225 layers, 11138696 parameters, 11138680 gradients, 28.7 GFLOPs
YOLOv8s summary: 225 layers, 11138696 parameters, 11138680 gradients, 28.7 GFLOPs


Freezing layer 'model.22.dfl.conv.weight'
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks with YOLOv8n...
AMP: running Automatic Mixed Precision (AMP) checks with YOLOv8n...


100%|██████████| 6.25M/6.25M [00:00<00:00, 83.7MB/s]
/usr/local/lib/python3.13/dist-packages/ultralytics/utils/checks.py:652: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(True):


...

  0%|          | 0/341 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/ultralytics/engine/trainer.py:374: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self.amp):
    150/150      1.99G      1.095     0.5837      1.031          6        640: 100%|██████████| 341/341 [00:24<00:00, 14.14it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:03<00:00, 14.30it/s]


                   all        682       1381       0.95      0.935       0.98      0.693
                   all        682       1381       0.95      0.935       0.98      0.693

150 epochs completed in 1.171 hours.

150 epochs completed in 1.171 hours.
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/yolov8s_seed1_150ep_auto/weights/last.pt, 22.5MB
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/yolov8s_seed1_150ep_auto/weights/last.pt, 22.5MB
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/yolov8s_seed1_150ep_auto/weights/best.pt, 22.5MB
Optimizer stripped from /content/drive/MyDrive/demamba-yolowd/output/yolov8s_seed1_150ep_auto/weights/best.pt, 22.5MB

Validating /content/drive/MyDrive/demamba-yolowd/output/yolov8s_seed1_150ep_auto/weights/best.pt...

Validating /content/drive/MyDrive/demamba-yolowd/output/yolov8s_seed1_150ep_auto/weights/best.pt...
Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVID

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:02<00:00, 17.33it/s]

                   all        682       1381       0.96      0.935       0.98      0.694
                   all        682       1381       0.96      0.935       0.98      0.694
              air-hole        406       1070       0.94      0.744      0.941      0.462
              air-hole        406       1070       0.94      0.744      0.941      0.462
             bite-edge          8          8       0.96          1      0.995      0.698
             bite-edge          8          8       0.96          1      0.995      0.698
            broken-arc         74         74      0.868      0.946      0.983      0.847
            broken-arc         74         74      0.868      0.946      0.983      0.847
                 crack         23         23          1      0.811      0.953      0.602
                 crack         23         23          1      0.811      0.953      0.602
        dense-gas-hole         43         43      0.962          1      0.995      0.831
        dense-gas-hol


SAVE_DIR: /content/drive/MyDrive/demamba-yolowd/output/yolov8s_seed1_150ep_auto
PENTING: cari baris 'optimizer:' di log & CATAT optimizer + lr yang dipilih framework.
final_eval selesai


## 9. Evaluasi + tulis CSV (berkas TERPISAH — tidak menimpa hasil lama)

In [ ]:
import pandas as pd
OUT_CSV = f"{DRIVE_ROOT}/analysis/{OUT_NAME}"
best = Path(SAVE_DIR)/"weights"/"best.pt"
assert best.exists(), "best.pt belum ada -- training belum tuntas"
r = YOLO(str(best)).val(data=DATA_YAML, split="val", verbose=False)

base = dict(model=MODEL_LABEL, seed=SEED, ckpt_path=str(best))
rows = [{**base, "class":"ALL", "mAP50_95":r.box.map, "mAP50":r.box.map50,
         "precision":r.box.mp, "recall":r.box.mr}]
for i, c in enumerate(NAMES):
    rows.append({**base, "class":c, "mAP50_95":float(r.box.maps[i]),
                 "mAP50":float("nan"), "precision":float("nan"), "recall":float("nan")})
df = pd.DataFrame(rows)
if os.path.exists(OUT_CSV):
    df = pd.concat([pd.read_csv(OUT_CSV), df]).drop_duplicates(["model","seed","class"], keep="last")
df.to_csv(OUT_CSV, index=False)
shutil.copy(OUT_CSV, f"/content/{OUT_NAME}")
print(df[df["class"]=="ALL"][["model","seed","mAP50_95","mAP50","precision","recall"]].to_string(index=False))
if AUTO:
    v = df[(df["class"]=="ALL") & (df.model==MODEL_LABEL)].mAP50_95.iloc[0]
    sgd = {"YOLOv3-tiny":0.5630,"YOLOv5s":0.6357,"YOLOv6n":0.5934,"YOLOv8n":0.5945,"YOLOv8s":0.6277}[MODEL_KEY]
    print(f"\n  {MODEL_KEY} resep terkunci (SGD 0.002) : {sgd:.4f}")
    print(f"  {MODEL_KEY} resep auto (RUN INI)       : {v:.4f}   ({v-sgd:+.4f})")
    print(f"  B2 auto (Mamba-YOLO-T)                 : 0.6796   (B2 terkunci naik +0.1098)")
    print(f"  -> {'B2-auto masih unggul' if 0.6796 > v else 'model ini MELAMPAUI B2-auto'}")
print(f"\nTersimpan: {OUT_CSV}\nKirim `{OUT_NAME}` ke sesi analisis.")
print(f"\nBerikutnya: ubah MODEL_KEY di Sel 3 -> {[k for k in MODELS if k != MODEL_KEY]}")

Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv8s summary (fused): 168 layers, 11128680 parameters, 0 gradients, 28.4 GFLOPs
YOLOv8s summary (fused): 168 layers, 11128680 parameters, 0 gradients, 28.4 GFLOPs


val: Scanning /content/dsf/steel-tube-dataset-all/yolo/labels/val2021.cache... 682 images, 2 backgrounds, 0 corrupt: 100%|██████████| 682/682 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:04<00:00,  9.44it/s]


                   all        682       1381       0.96      0.934      0.979      0.698
                   all        682       1381       0.96      0.934      0.979      0.698
Speed: 0.1ms preprocess, 2.0ms inference, 0.0ms loss, 1.2ms postprocess per image
Speed: 0.1ms preprocess, 2.0ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to runs/detect/val
Results saved to runs/detect/val
       model  seed  mAP50_95    mAP50  precision   recall
     B2-auto     1  0.679606 0.970531   0.951208 0.947711
YOLOv8n-auto     1  0.663947 0.967367   0.941636 0.936849
YOLOv8s-auto     1  0.697972 0.979487   0.959881 0.933910

  YOLOv8s resep terkunci (SGD 0.002) : 0.6277
  YOLOv8s resep auto (RUN INI)       : 0.6980   (+0.0703)
  B2 auto (Mamba-YOLO-T)                 : 0.6796   (B2 terkunci naik +0.1098)
  -> model ini MELAMPAUI B2-auto

Tersimpan: /content/drive/MyDrive/demamba-yolowd/analysis/per_class_results_recipe.csv
Kirim `per_class_results_recipe.csv` ke sesi analisis.


## 10. Matikan Runtime (Jalankan Terakhir)

In [ ]:
print("Semua proses selesai. Runtime akan di-disconnect otomatis...")
import time
time.sleep(10)
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"Gagal disconnect otomatis ({e})")
    print("Matikan manual lewat: Runtime > Disconnect and delete runtime")